# Symbolic Regression: Synthetic Pretraining + Feynman Eval

Train on synthetic univariate expressions, evaluate on both synthetic test set and Feynman equations.

In [1]:
# Environment setup — works on both Colab and local
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone repo if not already present
    REPO_DIR = '/content/symbolic-jepa'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zzpDavid2/symbolic-jepa.git {REPO_DIR}
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

    !pip install -q sympy scipy

    # Checkpoint dir on Drive for persistence
    CKPT_DIR = '/content/drive/MyDrive/Colab/checkpoints/synthetic_v1'
    LOG_DIR  = '/content/drive/MyDrive/Colab/runs'
else:
    CKPT_DIR = 'checkpoints/synthetic_v1'
    LOG_DIR  = 'runs'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
print(f'Environment: {"Colab" if IN_COLAB else "Local"}')
print(f'Checkpoints: {CKPT_DIR}')

Environment: Local
Checkpoints: checkpoints/synthetic_v1


In [2]:
import torch
import torch.nn.functional as F
import numpy as np
import datetime
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from symbolic_jepa import (
    PrefixTokenizer, Expression, load_feynman_csv,
    TNet, SymbolicTransformer,
    PointCloudDataset, build_feynman_splits, build_synthetic_splits,
    load_synthetic_pkl,
    teacher_forced_accuracy, evaluate_predictions,
)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Device: {DEVICE}')

/opt/anaconda3/envs/symba/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [5]:
# ── Hyperparameters ──
MAX_VARS    = 9
D_INPUT     = MAX_VARS + 1
N_POINTS    = 1000
MAX_SEQ     = 64
D_MODEL     = 512
N_HEADS     = 8
N_LAYERS    = 4
DROPOUT     = 0.2

EPOCHS      = 30
LR          = 3e-4
BATCH       = 16
VAL_EVERY   = 1
USE_AMP     = True

# Synthetic data (pre-generated by SYMBA_Reg_Data_Gen notebook)
SYNTH_PKL   = 'data/synthetic.pkl'   # path to pickle of expression strings
MAX_SYNTH   = 10_000                    # 0 = load all
SYNTH_SEED  = 42

## Load synthetic training data

In [4]:
tokenizer = PrefixTokenizer(max_vars=MAX_VARS)
print(f'Vocab size: {len(tokenizer)}')

print(f'Loading synthetic expressions from {SYNTH_PKL}...')
synth_exprs = load_synthetic_pkl(
    SYNTH_PKL, max_seq_len=MAX_SEQ,
    tokenizer=tokenizer, max_expressions=MAX_SYNTH,
)
print(f'Loaded {len(synth_exprs)} expressions')

# Inspect a few
for expr in synth_exprs[:5]:
    print(f'  {expr.prefix}')

Vocab size: 35
Loading synthetic expressions from data/s.pkl...


FileNotFoundError: [Errno 2] No such file or directory: 'data/s.pkl'

In [ ]:
# Split synthetic data
synth_train, synth_val, synth_test = build_synthetic_splits(
    synth_exprs, tokenizer,
    n_points=N_POINTS, max_seq_len=MAX_SEQ, max_vars=MAX_VARS,
    seed=SYNTH_SEED,
)

## Load Feynman test set

In [ ]:
CSV_PATH = 'data/FeynmanEquations.csv'
_, _, feynman_test = build_feynman_splits(
    CSV_PATH, tokenizer,
    n_points=N_POINTS, max_seq_len=MAX_SEQ, max_vars=MAX_VARS,
)
print(f'Feynman test set: {len(feynman_test)} equations')

## Build model

In [ ]:
encoder = TNet(d_input=D_INPUT, d_model=D_MODEL)
model = SymbolicTransformer(
    encoder=encoder,
    vocab_size=len(tokenizer),
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=4 * D_MODEL,
    max_seq_len=MAX_SEQ,
    dropout=DROPOUT,
    pad_id=tokenizer.pad_id,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Total parameters: {total_params:.1f}M')

## Training loop

In [ ]:
from torch.utils.tensorboard import SummaryWriter

run_name = f"synth_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
writer = SummaryWriter(log_dir=f'{LOG_DIR}/{run_name}')
print(f'TensorBoard: {LOG_DIR}/{run_name}')

In [ ]:
CKPT_PATH = f'{CKPT_DIR}/latest.pt'
BEST_PATH = f'{CKPT_DIR}/best.pt'

train_loader = DataLoader(synth_train, batch_size=BATCH, shuffle=True,
                          num_workers=2, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(synth_val, batch_size=BATCH, shuffle=False,
                          num_workers=2, persistent_workers=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch = 1
best_val    = float('inf')
history     = {'train': [], 'val': []}

# Resume from checkpoint if available
if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler'])
    start_epoch = ck['epoch'] + 1
    best_val    = ck['best_val']
    history     = ck['history']
    print(f'Resuming from epoch {start_epoch} (best val: {best_val:.4f})')

if USE_AMP and DEVICE == 'cuda':
    amp_ctx = lambda: torch.autocast('cuda', dtype=torch.bfloat16)
elif USE_AMP and DEVICE == 'mps':
    amp_ctx = lambda: torch.autocast('mps', dtype=torch.float16)
else:
    amp_ctx = lambda: torch.amp.autocast('cpu', enabled=False)

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in pbar:
        points    = batch['points'].to(DEVICE, non_blocking=True)
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        with amp_ctx():
            loss = model(points, input_ids, attn_mask=attn_mask)['loss']
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        global_step = (epoch - 1) * len(train_loader) + pbar.n
        writer.add_scalar('train/loss_step', loss.item(), global_step)

    scheduler.step()
    train_avg = train_loss / len(train_loader)
    history['train'].append(train_avg)
    writer.add_scalar('train/loss_epoch', train_avg, epoch)
    writer.add_scalar('train/lr', scheduler.get_last_lr()[0], epoch)

    # Validation
    if epoch % VAL_EVERY == 0 or epoch == EPOCHS:
        model.eval()
        val_loss = 0
        val_acc  = 0
        n_val_batches = 0
        with torch.no_grad(), amp_ctx():
            for batch in val_loader:
                points    = batch['points'].to(DEVICE, non_blocking=True)
                input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
                attn_mask = batch['attn_mask'].to(DEVICE, non_blocking=True)
                out = model(points, input_ids, attn_mask=attn_mask)
                val_loss += out['loss'].item()
                val_acc  += teacher_forced_accuracy(out['logits'], input_ids, tokenizer.pad_id)
                n_val_batches += 1

        val_avg = val_loss / n_val_batches
        val_acc_avg = val_acc / n_val_batches
        history['val'].append(val_avg)
        history.setdefault('val_acc', []).append(val_acc_avg)
        writer.add_scalar('val/loss', val_avg, epoch)
        writer.add_scalar('val/token_accuracy', val_acc_avg, epoch)

        is_best = val_avg < best_val
        if is_best: best_val = val_avg
        flag = ' ★ best' if is_best else ''
        print(f'Epoch {epoch}/{EPOCHS} | train={train_avg:.4f} | val={val_avg:.4f}{flag} | val_acc={val_acc_avg*100:.1f}%')

        if is_best:
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'val': val_avg}, BEST_PATH)
    else:
        print(f'Epoch {epoch}/{EPOCHS} | train={train_avg:.4f}')

    torch.save({
        'model':     model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'epoch':     epoch,
        'best_val':  best_val,
        'history':   history,
    }, CKPT_PATH)

writer.close()
print('Training complete.')

## Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train'], label='train')
ax1.plot(range(VAL_EVERY, len(history['train']) + 1, VAL_EVERY)[:len(history['val'])],
         history['val'], label='val', marker='o')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss')
ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('Loss')

if 'val_acc' in history:
    ax2.plot(range(VAL_EVERY, len(history['train']) + 1, VAL_EVERY)[:len(history['val_acc'])],
             [a * 100 for a in history['val_acc']], marker='o', color='green')
    ax2.set_xlabel('epoch'); ax2.set_ylabel('token accuracy %')
    ax2.grid(alpha=0.3)
    ax2.set_title('Val Token Accuracy')

plt.tight_layout()
plt.show()

## Evaluate on synthetic test set

In [ ]:
# Load best checkpoint
if os.path.exists(BEST_PATH):
    ck = torch.load(BEST_PATH, map_location=DEVICE)
    model.load_state_dict(ck['model'])
    print(f'Loaded best checkpoint (epoch {ck["epoch"]}, val={ck["val"]:.4f})')

def run_eval(dataset, name, beam_width=10):
    """Run greedy + beam eval on a dataset, print summary."""
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False)
    model.eval()

    # Greedy
    greedy_preds = []
    for batch in loader:
        points = batch['points'].to(DEVICE)
        input_ids = batch['input_ids']
        preds = model.generate(points, tokenizer, max_new_tokens=MAX_SEQ)
        for j, pred_str in enumerate(preds):
            gt_str = tokenizer.decode(input_ids[j].tolist())
            greedy_preds.append((gt_str, pred_str))

    greedy_results = evaluate_predictions(greedy_preds, dataset, tokenizer)
    print(f'\n── {name} (Greedy) ──')
    print(f'  Exact match:     {greedy_results["exact_match"]*100:.1f}%')
    print(f'  Token accuracy:  {greedy_results["token_accuracy"]*100:.1f}%')
    print(f'  Algebraic equiv: {greedy_results["algebraic_equiv"]*100:.1f}%')
    print(f'  Mean R²:         {greedy_results["mean_r2"]:.4f}')
    print(f'  R² > 0.9:        {greedy_results["r2_above_0.9"]*100:.1f}%')

    # Beam search
    beam_preds = []
    for batch in tqdm(loader, desc=f'{name} beam search'):
        points = batch['points'].to(DEVICE)
        input_ids = batch['input_ids']
        for j in range(points.shape[0]):
            pred = model.generate_beam(
                points[j:j+1], tokenizer,
                max_new_tokens=MAX_SEQ, beam_width=beam_width,
            )[0]
            gt_str = tokenizer.decode(input_ids[j].tolist())
            beam_preds.append((gt_str, pred))

    beam_results = evaluate_predictions(beam_preds, dataset, tokenizer)
    print(f'\n── {name} (Beam width={beam_width}) ──')
    print(f'  Exact match:     {beam_results["exact_match"]*100:.1f}%')
    print(f'  Token accuracy:  {beam_results["token_accuracy"]*100:.1f}%')
    print(f'  Algebraic equiv: {beam_results["algebraic_equiv"]*100:.1f}%')
    print(f'  Mean R²:         {beam_results["mean_r2"]:.4f}')
    print(f'  R² > 0.9:        {beam_results["r2_above_0.9"]*100:.1f}%')

    return greedy_results, beam_results

In [ ]:
synth_greedy, synth_beam = run_eval(synth_test, 'Synthetic Test')

## Evaluate on Feynman test set

In [ ]:
feyn_greedy, feyn_beam = run_eval(feynman_test, 'Feynman Test')

## Inspect predictions

In [ ]:
# Show a few beam-search predictions from each test set
for name, results in [('Synthetic', synth_beam), ('Feynman', feyn_beam)]:
    print(f'\n═══ {name} predictions (first 10) ═══')
    for d in results['details'][:10]:
        print(f'  GT:   {d["gt"]}')
        print(f'  Pred: {d["pred"]}')
        r2_str = f'{d["r2"]:.4f}' if d['r2'] is not None else 'N/A'
        print(f'  R²:   {r2_str}  | exact={bool(d["exact"])}')
        print()